In [ ]:
import os

base = r'C:\SERCOP_FINAL\05_documentos'
print("Contenido de 05_documentos:")
for item in os.listdir(base):
    print(" -", item)

# Búsqueda recursiva por si el archivo está en una subcarpeta con otro nombre
print("\nBúsqueda de archivos que contengan 'compan' o 'directorio' (insensible a mayúsculas):")
for root, dirs, files in os.walk(r'C:\SERCOP_FINAL'):
    for f in files:
        if 'compan' in f.lower() or 'directorio' in f.lower():
            print(os.path.join(root, f))

In [ ]:
!pip install openpyxl

In [ ]:
import pandas as pd

archivo = r'C:\SERCOP_FINAL\05_documentos\directorio_companias.xlsx'
df_directorio = pd.read_excel(archivo, nrows=20)
print("Columnas:", df_directorio.columns.tolist())
print("\nMuestra:")
print(df_directorio.head(10))
print("\nTipos:")
print(df_directorio.dtypes)

import openpyxl
wb = openpyxl.load_workbook(archivo, read_only=True)
ws = wb.active
print("\nFilas totales (aprox):", ws.max_row, "| Columnas:", ws.max_column)
print("Hojas del archivo:", wb.sheetnames)

In [ ]:
import pandas as pd

archivo = r'C:\SERCOP_FINAL\05_documentos\directorio_companias.xlsx'
directorio = pd.read_excel(archivo, header=4)
print("Columnas:", directorio.columns.tolist())
print("Filas:", len(directorio))
print(directorio.head(3))

In [ ]:
import os
os.chdir(r'C:\SERCOP_FINAL')
print(os.getcwd())

In [ ]:
directorio_ref.to_parquet('03_resultados/directorio_companias.parquet', index=False)
print("RUC únicos en directorio:", directorio_ref['RUC'].nunique())
print("Nulos en REPRESENTANTE:", directorio_ref['REPRESENTANTE'].isna().sum())

In [ ]:
directorio['RUC'] = directorio['RUC'].astype(str).str.strip().str.zfill(13)
directorio_ref = directorio[['RUC','NOMBRE','SITUACIÓN LEGAL','TIPO','REPRESENTANTE','CARGO']].drop_duplicates('RUC')
directorio_ref.to_parquet('03_resultados/directorio_companias.parquet', index=False)
print("RUC únicos en directorio:", directorio_ref['RUC'].nunique())
print("Nulos en REPRESENTANTE:", directorio_ref['REPRESENTANTE'].isna().sum())

import duckdb
con2 = duckdb.connect()
con2.execute("CREATE OR REPLACE TABLE dominancia_pares AS SELECT * FROM read_parquet('03_resultados/dominancia_pares.parquet')")
con2.execute("CREATE OR REPLACE TABLE directorio_ref AS SELECT * FROM directorio_ref")

cruce_dominancia = con2.execute("""
    SELECT p.ruc_a, p.ruc_b, p.coincidencias, p.dominancia_max,
           da.NOMBRE AS nombre_a, da.REPRESENTANTE AS representante_a,
           db.NOMBRE AS nombre_b, db.REPRESENTANTE AS representante_b,
           (da.REPRESENTANTE IS NOT NULL AND da.REPRESENTANTE = db.REPRESENTANTE) AS mismo_representante
    FROM dominancia_pares p
    LEFT JOIN directorio_ref da ON p.ruc_a = da.RUC
    LEFT JOIN directorio_ref db ON p.ruc_b = db.RUC
    WHERE p.dominancia_max >= 0.6
    ORDER BY p.dominancia_max DESC
""").df()

print("\nPares con dominancia_max >= 0.6:", len(cruce_dominancia))
print("De esos, con AMBOS RUC identificados en el directorio (tipo SOCIEDAD):",
      cruce_dominancia[['representante_a','representante_b']].notna().all(axis=1).sum())
print("Pares que comparten representante legal:", cruce_dominancia['mismo_representante'].sum())

con2.execute("CREATE OR REPLACE TABLE base_modelado_v14 AS SELECT * FROM read_parquet('03_resultados/base_modelado_v14.parquet')")
fantasmas_ruc = con2.execute("SELECT DISTINCT proveedor_ruc FROM base_modelado_v14 WHERE proveedor_es_fantasma = TRUE").df()['proveedor_ruc'].tolist()
cruce_dominancia['involucra_fantasma'] = cruce_dominancia['ruc_a'].isin(fantasmas_ruc) | cruce_dominancia['ruc_b'].isin(fantasmas_ruc)
print("\nPares con dominancia alta que involucran un fantasma conocido:", cruce_dominancia['involucra_fantasma'].sum())
print(cruce_dominancia[cruce_dominancia['involucra_fantasma']][['ruc_a','ruc_b','dominancia_max','nombre_a','representante_a','nombre_b','representante_b','mismo_representante']].to_string(index=False))

# También cruzamos si el proveedor fantasma en sí está en el directorio (para saber si es SOCIEDAD, útil para interpretar el resultado)
fantasmas_en_directorio = con2.execute("""
    SELECT DISTINCT b.proveedor_ruc, d.NOMBRE, d.TIPO, d.SITUACION_LEGAL, d.REPRESENTANTE
    FROM base_modelado_v14 b
    LEFT JOIN directorio_ref d ON b.proveedor_ruc = d.RUC
    WHERE b.proveedor_es_fantasma = TRUE
""").df()
print("\nLos 15 fantasmas: ¿están en el directorio de Superintendencia (son tipo SOCIEDAD)?")
print(fantasmas_en_directorio.to_string(index=False))

In [ ]:
mismos_representantes = cruce_dominancia[cruce_dominancia['mismo_representante'] == True]
print("Los 9 pares con mismo representante legal y dominancia alta:")
print(mismos_representantes[['ruc_a','ruc_b','coincidencias','dominancia_max','nombre_a','nombre_b','representante_a']].to_string(index=False))

# Contexto: en qué entidades/procesos ganaron estos pares, y si alguno tiene otras señales (Benford, entrada a mercado reciente, etc.)
rucs_interes = pd.unique(mismos_representantes[['ruc_a','ruc_b']].values.ravel())
contexto = con2.execute(f"""
    SELECT proveedor_ruc, entidad_ruc, anio_proceso, monto_final, tipo_contratacion_canonico,
           categoria_principal
    FROM base_modelado_v14
    WHERE proveedor_ruc IN ({','.join([repr(r) for r in rucs_interes])})
    ORDER BY proveedor_ruc, anio_proceso
""").df()
print(f"\nTotal de procesos de estos {len(rucs_interes)} proveedores:", len(contexto))
print(contexto.groupby('proveedor_ruc').agg(n_procesos=('anio_proceso','size'),
                                              monto_total=('monto_final','sum'),
                                              anio_min=('anio_proceso','min'),
                                              anio_max=('anio_proceso','max')))

In [ ]:
# Sin filtro de dominancia: ¿cuántos pares en TODO el universo (327,266) comparten representante legal?
cruce_completo = con2.execute("""
    SELECT p.ruc_a, p.ruc_b, p.coincidencias, p.dominancia_max,
           da.NOMBRE AS nombre_a, da.REPRESENTANTE AS representante_a,
           db.NOMBRE AS nombre_b, db.REPRESENTANTE AS representante_b
    FROM dominancia_pares p
    LEFT JOIN directorio_ref da ON p.ruc_a = da.RUC
    LEFT JOIN directorio_ref db ON p.ruc_b = db.RUC
    WHERE da.REPRESENTANTE IS NOT NULL AND da.REPRESENTANTE = db.REPRESENTANTE
    ORDER BY p.dominancia_max DESC
""").df()

print("Total de pares (de 327,266) que comparten representante legal, cualquier nivel de dominancia:", len(cruce_completo))
print("\nDistribución de dominancia_max en estos pares:")
print(cruce_completo['dominancia_max'].describe())
print("\n% con dominancia_max >= 0.6:", (cruce_completo['dominancia_max']>=0.6).mean()*100, "%")

# Comparar contra la tasa base: ¿qué % de TODOS los pares (no solo los que comparten representante) tiene dominancia >= 0.6?
tasa_base = con2.execute("SELECT AVG(CASE WHEN dominancia_max >= 0.6 THEN 1.0 ELSE 0.0 END) AS tasa FROM dominancia_pares").df()
print("\nTasa base de dominancia>=0.6 en TODOS los pares:", tasa_base['tasa'].iloc[0]*100, "%")
print("=> Lift de compartir representante legal sobre dominancia alta:",
      (cruce_completo['dominancia_max']>=0.6).mean() / tasa_base['tasa'].iloc[0])

cruce_completo.to_parquet('03_resultados/pares_mismo_representante.parquet', index=False)
print("\nTop 15 por dominancia:")
print(cruce_completo[['nombre_a','nombre_b','coincidencias','dominancia_max']].head(15).to_string(index=False))

In [ ]:
con2.execute("CREATE OR REPLACE TABLE oferentes_consolidado AS SELECT * FROM read_parquet('03_resultados/oferentes_consolidado.parquet')")

In [ ]:
pares_top9 = mismos_representantes[mismos_representantes['dominancia_max'] >= 0.6][['ruc_a','ruc_b']]

detalle_casos = []
for _, row in pares_top9.iterrows():
    ra, rb = row['ruc_a'], row['ruc_b']
    detalle = con2.execute(f"""
        WITH procesos_par AS (
            SELECT DISTINCT o.ocid
            FROM oferentes_consolidado o
            WHERE o.oferente_ruc IN ('{ra}', '{rb}')
            GROUP BY o.ocid
            HAVING COUNT(DISTINCT o.oferente_ruc) = 2
        )
        SELECT b.ocid, b.entidad_ruc, b.anio_proceso, b.fecha_release, b.monto_final,
               b.proveedor_ruc AS ganador_ruc, b.tipo_contratacion_canonico
        FROM base_modelado_v14 b
        JOIN procesos_par p ON b.ocid = p.ocid
        ORDER BY b.fecha_release
    """).df()
    detalle['par'] = f"{ra} / {rb}"
    detalle_casos.append(detalle)

detalle_final = pd.concat(detalle_casos, ignore_index=True)
detalle_final.to_csv('03_resultados/detalle_9_casos_representante.csv', index=False, encoding='utf-8-sig')
print("Filas:", len(detalle_final))
print(detalle_final[['par','ocid','entidad_ruc','anio_proceso','monto_final','ganador_ruc','tipo_contratacion_canonico']].to_string(index=False))

In [ ]:
pares_35 = mismos_representantes if 'mismos_representantes' in dir() else cruce_completo  # usa la tabla de los 35 pares completos

detalle_35 = []
for _, row in pares_35[['ruc_a','ruc_b']].drop_duplicates().iterrows():
    ra, rb = row['ruc_a'], row['ruc_b']
    d = con2.execute(f"""
        WITH procesos_par AS (
            SELECT o.ocid FROM oferentes_consolidado o
            WHERE o.oferente_ruc IN ('{ra}', '{rb}')
            GROUP BY o.ocid HAVING COUNT(DISTINCT o.oferente_ruc) = 2
        )
        SELECT b.entidad_ruc FROM base_modelado_v14 b JOIN procesos_par p ON b.ocid = p.ocid
    """).df()
    d['par'] = f"{ra} / {rb}"
    detalle_35.append(d)

detalle_35_final = pd.concat(detalle_35, ignore_index=True)

# ¿Qué entidades aparecen en MÁS DE UN par distinto?
entidades_por_par = detalle_35_final.groupby('entidad_ruc')['par'].nunique().sort_values(ascending=False)
entidades_repetidas = entidades_por_par[entidades_por_par > 1]
print("Entidades que aparecen en más de un par 'mismo representante':")
print(entidades_repetidas)

print("\nDetalle de esas entidades y con qué pares se cruzan:")
for entidad in entidades_repetidas.index:
    pares_de_entidad = detalle_35_final[detalle_35_final['entidad_ruc']==entidad]['par'].unique()
    print(f"\n{entidad}:")
    for p in pares_de_entidad:
        print("  -", p)

In [ ]:
entidades_interes = ['1768152800001', '1360051890001', '2360006130001']

perfil_entidades = con2.execute(f"""
    WITH stats_entidad AS (
        SELECT entidad_ruc,
               COUNT(*) AS n_procesos,
               SUM(monto_final) AS monto_total,
               COUNT(DISTINCT proveedor_ruc) AS n_proveedores_distintos,
               AVG(CASE WHEN proveedor_unico = 1 THEN 1.0 ELSE 0.0 END) AS pct_proveedor_unico,
               MIN(anio_proceso) AS anio_min, MAX(anio_proceso) AS anio_max
        FROM base_modelado_v14
        WHERE proveedor_ruc IS NOT NULL
        GROUP BY entidad_ruc
    ),
    top_proveedor AS (
        SELECT entidad_ruc, proveedor_ruc, SUM(monto_final) AS monto_proveedor,
               ROW_NUMBER() OVER (PARTITION BY entidad_ruc ORDER BY SUM(monto_final) DESC) AS rn
        FROM base_modelado_v14
        WHERE proveedor_ruc IS NOT NULL
        GROUP BY entidad_ruc, proveedor_ruc
    )
    SELECT s.entidad_ruc, s.n_procesos, s.monto_total, s.n_proveedores_distintos, s.pct_proveedor_unico,
           s.anio_min, s.anio_max,
           t.monto_proveedor AS monto_top_proveedor,
           t.monto_proveedor / s.monto_total AS concentracion_top_proveedor
    FROM stats_entidad s
    JOIN top_proveedor t ON s.entidad_ruc = t.entidad_ruc AND t.rn = 1
    WHERE s.entidad_ruc IN ({','.join([repr(e) for e in entidades_interes])})
""").df()
print("Perfil de las 3 entidades de interés:")
print(perfil_entidades.to_string(index=False))

# Comparación contra el resto de entidades (promedios/medianas de referencia)
referencia = con2.execute("""
    WITH stats_entidad AS (
        SELECT entidad_ruc,
               COUNT(*) AS n_procesos,
               COUNT(DISTINCT proveedor_ruc) AS n_proveedores_distintos,
               AVG(CASE WHEN proveedor_unico = 1 THEN 1.0 ELSE 0.0 END) AS pct_proveedor_unico
        FROM base_modelado_v14
        WHERE proveedor_ruc IS NOT NULL
        GROUP BY entidad_ruc
    ),
    monto_por_proveedor AS (
        SELECT entidad_ruc, proveedor_ruc, SUM(monto_final) AS monto_proveedor
        FROM base_modelado_v14 WHERE proveedor_ruc IS NOT NULL
        GROUP BY entidad_ruc, proveedor_ruc
    ),
    top_proveedor AS (
        SELECT entidad_ruc, SUM(monto_proveedor) AS monto_total_e, MAX(monto_proveedor) AS monto_top
        FROM monto_por_proveedor
        GROUP BY entidad_ruc
    )
    SELECT MEDIAN(s.pct_proveedor_unico) AS mediana_pct_proveedor_unico,
           MEDIAN(t.monto_top / t.monto_total_e) AS mediana_concentracion_top,
           MEDIAN(s.n_proveedores_distintos) AS mediana_n_proveedores
    FROM stats_entidad s JOIN top_proveedor t ON s.entidad_ruc = t.entidad_ruc
    WHERE s.n_procesos >= 10
""").df()
print("Referencia (mediana de TODAS las entidades con >=10 procesos):")
print(referencia)

In [ ]:
import os
for root, dirs, files in os.walk(r'C:\SERCOP_FINAL'):
    for f in files:
        if 'catastro' in f.lower() or 'sri_ruc' in f.lower():
            print(os.path.join(root, f))

In [ ]:
import duckdb, pandas as pd

con2 = duckdb.connect()
con2.execute("CREATE OR REPLACE TABLE base_modelado_v14 AS SELECT * FROM read_parquet('03_resultados/base_modelado_v14.parquet')")
con2.execute("CREATE OR REPLACE TABLE sri_ruc_catastro AS SELECT * FROM read_parquet('03_resultados/sri_ruc_catastro.parquet')")
con2.execute("CREATE OR REPLACE TABLE directorio_ref AS SELECT * FROM read_parquet('03_resultados/directorio_companias.parquet')")

print("Columnas SRI catastro:", con2.execute("DESCRIBE sri_ruc_catastro").df()['column_name'].tolist())
print("RUC únicos SRI catastro:", con2.execute("SELECT COUNT(DISTINCT NUMERO_RUC) FROM sri_ruc_catastro").df().iloc[0,0])

print("\nColumnas Superintendencia:", con2.execute("DESCRIBE directorio_ref").df()['column_name'].tolist())
print("RUC únicos Superintendencia:", con2.execute("SELECT COUNT(DISTINCT RUC) FROM directorio_ref").df().iloc[0,0])

cols_base = con2.execute("DESCRIBE base_modelado_v14").df()['column_name'].tolist()
print("\n¿'proveedor_es_fantasma' en base_modelado_v14?:", 'proveedor_es_fantasma' in cols_base)
print("¿'entidad_es_fantasma' en base_modelado_v14?:", 'entidad_es_fantasma' in cols_base)
print("Total columnas actuales:", len(cols_base))
print(cols_base)

In [ ]:
con2.execute("DROP TABLE IF EXISTS directorio_ref")
con2.execute("CREATE TABLE directorio_ref AS SELECT * FROM directorio")  # 'directorio' es el DataFrame completo original (25 columnas), directorio_ref es el subset en pandas
print(con2.execute("DESCRIBE directorio_ref").df()['column_name'].tolist())

In [ ]:
import pandas as pd

# 1) Reconstruir directorio_ref con más columnas útiles
archivo = r'C:\SERCOP_FINAL\05_documentos\directorio_companias.xlsx'
directorio = pd.read_excel(archivo, header=4)
directorio['RUC'] = directorio['RUC'].astype(str).str.strip().str.zfill(13)
directorio_ref = directorio[['RUC','NOMBRE','SITUACIÓN LEGAL','TIPO','FECHA_CONSTITUCION',
                              'REPRESENTANTE','CARGO','PROVINCIA','CAPITAL SUSCRITO']].drop_duplicates('RUC')
directorio_ref.to_parquet('03_resultados/directorio_companias.parquet', index=False)
print("directorio_ref reconstruido:", len(directorio_ref), "filas,", directorio_ref.columns.tolist())

# 2) Base maestra: agregar SRI (lado entidad) + Superintendencia (ambos lados)
con2.execute("CREATE OR REPLACE TABLE directorio_ref AS SELECT * FROM directorio_ref")

base_maestra = con2.execute("""
    SELECT b.*,
           -- SRI lado entidad (espejo de lo que ya existe para proveedor)
           se.ESTADO_CONTRIBUYENTE AS entidad_estado_actual_sri,
           se.FECHA_INICIO_ACTIVIDADES AS entidad_fecha_inicio_actividades_sri,
           se.FECHA_SUSPENSION_DEFINITIVA AS entidad_fecha_suspension_definitiva_sri,
           se.FECHA_REINICIO_ACTIVIDADES AS entidad_fecha_reinicio_actividades_sri,
           -- Superintendencia lado proveedor
           dp."SITUACIÓN LEGAL" AS proveedor_superintendencia_situacion,
           dp.TIPO AS proveedor_superintendencia_tipo,
           dp.FECHA_CONSTITUCION AS proveedor_superintendencia_fecha_constitucion,
           dp.REPRESENTANTE AS proveedor_superintendencia_representante,
           -- Superintendencia lado entidad
           de."SITUACIÓN LEGAL" AS entidad_superintendencia_situacion,
           de.TIPO AS entidad_superintendencia_tipo,
           de.FECHA_CONSTITUCION AS entidad_superintendencia_fecha_constitucion,
           de.REPRESENTANTE AS entidad_superintendencia_representante
    FROM base_modelado_v14 b
    LEFT JOIN sri_ruc_catastro se ON b.entidad_ruc = se.NUMERO_RUC
    LEFT JOIN directorio_ref dp ON b.proveedor_ruc = dp.RUC
    LEFT JOIN directorio_ref de ON b.entidad_ruc = de.RUC
""").df()

print("\nbase_maestra:", len(base_maestra), "filas,", len(base_maestra.columns), "columnas")
base_maestra.to_parquet('03_resultados/base_maestra_v1.parquet', index=False)
print("Guardada en 03_resultados/base_maestra_v1.parquet")

# Cobertura de los nuevos cruces (cuánto realmente encontró match)
print("\nCobertura de los nuevos campos (% no nulo):")
nuevas_cols = ['entidad_estado_actual_sri','proveedor_superintendencia_situacion','entidad_superintendencia_situacion']
for c in nuevas_cols:
    print(f"  {c}: {base_maestra[c].notna().mean()*100:.2f}%")

In [ ]:
print(con2.execute("DESCRIBE directorio_ref").df()['column_name'].tolist())

In [ ]:
auditoria_nulos = pd.DataFrame({
    'columna': base_maestra.columns,
    'pct_nulo': [base_maestra[c].isna().mean()*100 for c in base_maestra.columns],
    'tipo_dato': [str(base_maestra[c].dtype) for c in base_maestra.columns]
}).sort_values('pct_nulo', ascending=False)

pd.set_option('display.max_rows', 100)
print(auditoria_nulos.to_string(index=False))

auditoria_nulos.to_csv('03_resultados/auditoria_nulos_base_maestra.csv', index=False, encoding='utf-8-sig')
print("\nGuardado en 03_resultados/auditoria_nulos_base_maestra.csv")

In [ ]:
base_maestra['proveedor_fue_suspendido_alguna_vez'] = base_maestra['proveedor_fecha_suspension_definitiva'].notna()
base_maestra['entidad_fue_suspendida_alguna_vez'] = base_maestra['entidad_fecha_suspension_definitiva_sri'].notna()
base_maestra['proveedor_es_sociedad_registrada'] = base_maestra['proveedor_superintendencia_situacion'].notna()
base_maestra['entidad_es_sociedad_registrada'] = base_maestra['entidad_superintendencia_situacion'].notna()

print("Nuevas banderas creadas:")
for col in ['proveedor_fue_suspendido_alguna_vez','entidad_fue_suspendida_alguna_vez',
            'proveedor_es_sociedad_registrada','entidad_es_sociedad_registrada']:
    print(f"  {col}: {base_maestra[col].mean()*100:.2f}% True")

base_maestra.to_parquet('03_resultados/base_maestra_v2.parquet', index=False)
print("\nbase_maestra_v2 guardada:", len(base_maestra), "filas,", len(base_maestra.columns), "columnas")

# Diccionario de datos completo (para compartir con el tutor)
diccionario = pd.DataFrame({
    'columna': base_maestra.columns,
    'tipo_dato': [str(base_maestra[c].dtype) for c in base_maestra.columns],
    'pct_nulo': [round(base_maestra[c].isna().mean()*100, 3) for c in base_maestra.columns]
})
diccionario.to_csv('03_resultados/diccionario_datos_base_maestra.csv', index=False, encoding='utf-8-sig')
print("Diccionario de datos guardado en 03_resultados/diccionario_datos_base_maestra.csv")